# Uncertainty-TTA on Kaggle / Colab — Phases 1–5

Baseline training, standard TTA, uncertainty-weighted TTA, **and the Phase 5
additions** (EfficientNet-B0 backbone, Top-K TTA, inference benchmark, one-command
`run_all`), all driven by the repo's `scripts/` so results match local runs exactly.

**Kaggle data handling contributed by S2 (Mohamed Abdel Sattar).** This notebook
expects the repo to be attached as a **Kaggle Dataset** (folder
`uncertainty-tta-medmnist-main`) — the setup cell finds it under `/kaggle/input`,
copies it into the writable working dir, and clears any cached `src` imports so
code edits take effect without a kernel restart. See `docs/KAGGLE_SETUP.md`.

> ⚠️ **Phase 5 update:** re-upload the *current* repo as your Kaggle Dataset before
> running the Phase 5 cells — they need the new `--arch effb0`, Top-K, and
> `scripts/run_all.py` code. If your attached Dataset is the old Phase-3 snapshot,
> the EfficientNet cells will fail with "unrecognized arguments: --arch".

> All fusion math runs through the repo scripts → the repo is the single source of
> truth. Generate every tracker number here, don't paste from a standalone notebook,
> so every student's matrix row stays comparable.

**This notebook is set up for DermaMNIST (S2's dataset).** Change `DATASET` below to
reuse it for another modality.

## 1. Verify GPU

In [ ]:
import torch
print('PyTorch    :', torch.__version__)
print('CUDA avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU        :', torch.cuda.get_device_name(0))
else:
    print('⚠ No GPU detected. Enable it in Settings → Accelerator before continuing.')

## 2. Attach the repo + install dependencies

`torch`/`torchvision` are preinstalled on Kaggle GPU images; we only add `medmnist`.
The cell after the install is **S2's** Kaggle data-handler — it locates the repo
Dataset under `/kaggle/input`, copies it to the writable working dir, `chdir`s in,
and drops cached `src.*` modules so edits take effect without a restart.

In [ ]:
!pip install -q medmnist==3.0.2 2>/dev/null

In [ ]:
from pathlib import Path
import shutil
import os
import sys

# =========================================================
# Change these only
# =========================================================
PROJECT_FOLDER_NAME = "uncertainty-tta-medmnist-main"

# =========================================================
# Kaggle roots
# =========================================================
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")

# =========================================================
# Find project folder inside /kaggle/input
# =========================================================
project_matches = [p for p in INPUT_ROOT.rglob(PROJECT_FOLDER_NAME) if p.is_dir()]

if not project_matches:
    raise FileNotFoundError(f"Could not find project folder: {PROJECT_FOLDER_NAME}")

project_src = project_matches[0]
project_dst = WORKING_ROOT / PROJECT_FOLDER_NAME

print("Project source:", project_src)
print("Project destination:", project_dst)

# =========================================================
# Copy project to /kaggle/working
# =========================================================
if project_dst.exists():
    shutil.rmtree(project_dst)

shutil.copytree(project_src, project_dst)

# =========================================================
# Move into project folder
# =========================================================
os.chdir(project_dst)

# =========================================================
# Remove old cached src imports
# =========================================================
for m in list(sys.modules):
    if m == "src" or m.startswith("src."):
        del sys.modules[m]

# =========================================================
# Add project root to Python path
# =========================================================
PROJECT_DIR = str(project_dst)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# =========================================================
# Confirm setup
# =========================================================
print("\nCurrent directory:")
print(os.getcwd())

print("\nPython path:")
print(sys.path[:3])

print("\nProject files:")
print(os.listdir("."))

## 3. (Optional) Visual sanity check — preview 6 training samples

In [ ]:
from src.data import get_dataset
from src.config import get_config
import matplotlib.pyplot as plt

DATASET = 'dermamnist'
cfg = get_config(DATASET)
train_ds = get_dataset(cfg, split='train', img_size=64)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def denorm(t):
    t = t.clone()
    for c in range(3):
        t[c] = t[c] * IMAGENET_STD[c] + IMAGENET_MEAN[c]
    return t.clamp(0, 1)

fig, axes = plt.subplots(1, 6, figsize=(12, 2.2))
for i, ax in enumerate(axes):
    img, lbl = train_ds[i]
    ax.imshow(denorm(img).permute(1, 2, 0).numpy())
    ax.set_title(f'y={int(lbl.squeeze())}', fontsize=9)
    ax.axis('off')
plt.suptitle(f'{cfg.modality} — first 6 train samples', y=1.05, fontsize=10)
plt.tight_layout(); plt.show()
print(f'train / val / test sizes will be loaded by the script below.')

## 4. Choose the dataset + seeds

Everything below keys off these. DermaMNIST is S2's assignment; the three seeds give
the mean±std the paper reports.

In [ ]:
DATASET = 'dermamnist'      # S2's dataset; change to run another modality
SEEDS   = [0, 42, 123]       # 3-seed stability study
NWORK   = 2                  # Kaggle is Linux, so workers>0 is safe (faster than 0)
print('Dataset:', DATASET, '| seeds:', SEEDS)

## 5. ResNet-18 — baseline + weighted TTA (Phases 1–3)

This is S2's original Phase 1–3 pipeline (ResNet-18). It's already committed, so
**only rerun this to reproduce** the canonical numbers. Each seed trains 30 epochs,
runs all 8 strategies **plus the new Top-3/5/7 columns** (Top-K now runs by default
in `run_weighted_tta`), then `aggregate_seeds` writes the mean±std table.

`--no-time` skips the latency pass here (we measure latency once, separately, in
section 8).

In [ ]:
for s in SEEDS:
    tag = f'_seed{s}'   # tag ALL seeds (incl 42) so aggregate_seeds globs them; matches S2's original flow
    !python -m scripts.train_baseline   --dataset {DATASET} --seed {s} --ckpt-tag {tag} --num-workers {NWORK}
    !python -m scripts.run_weighted_tta --dataset {DATASET} --ckpt-tag {tag} --seed {s} --no-time --num-workers {NWORK}
!python -m scripts.aggregate_seeds --datasets {DATASET}

## 6. Phase 5 — EfficientNet-B0, the whole pipeline in one command (NEW)

`scripts/run_all.py` runs every phase for a backbone × dataset × seed in one go:
trains EfficientNet-B0 for all 3 seeds, runs weighted TTA (8 strategies + Top-K),
standard TTA, both ablations, the **confidence strips** (DermaMNIST is one of the
Figure-1 modalities), the **inference benchmark**, and finally `aggregate`.

Notes:
- EfficientNet has no untagged canonical checkpoint, so `run_all` auto-copies the
  seed-42 checkpoint into place for the standard/ablate/strips/benchmark phases —
  you'll see `[prep] effb0/dermamnist: copied ...` lines. No manual step needed.
- The "±2% vs ResNet-18 benchmark" line on effb0 trainings is **informational**, not
  a failure — a different backbone isn't expected to hit ResNet's exact number.
- `--continue-on-error` keeps going if any single step hiccups.

DermaMNIST is small (~7k train images), so all of this is comfortably within one
Kaggle GPU session.

In [ ]:
!python -m scripts.run_all \
    --models effb0 --seeds 0 42 123 --datasets {DATASET} \
    --phases train weighted standard ablate strips benchmark aggregate \
    --num-workers {NWORK} --continue-on-error

## 7. EfficientNet-B0 results — weighted TTA + seed stability

The `arch` column distinguishes backbones. Top-3/5/7 appear after the 8 core
strategies.

In [ ]:
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)

wt = pd.read_csv(f'results/{DATASET}_effb0_weighted_tta.csv')
print('=== EfficientNet-B0 weighted TTA (seed 123 shown; one file per seed) ===')
display(wt[['arch','strategy','accuracy','ece','nll','auc_roc','inf_ms']].round(4))

print('\n=== EfficientNet-B0 mean ± std across seeds ===')
agg = pd.read_csv('results/effb0_seed_stability.csv')
display(agg[agg.dataset == DATASET].round(4))

## 8. Phase 5 — variance sanity check + inference benchmark

The variance check proves the proposal's `w = 1/(var+ε)` is backward and that the
repo's confidence-aligned `w = var` is correct (exit 0 = code matches). The benchmark
table (already generated by `run_all` above) shows latency across N and method — every
soft strategy and every Top-K share the `tta_all_strategies` row at each N.

In [ ]:
!python -m scripts.variance_sanity_check

import pandas as pd
bench = pd.read_csv('results/inference_benchmark.csv')
print('\n=== Inference benchmark (EfficientNet-B0) ===')
display(bench[bench.dataset == DATASET].round(4))

## 9. Figure 1 — Augmentation Confidence Strip (gold = Top-5 kept views)

`run_all` produced these in `figures/strip/`. The gold-outlined bars are the Top-K
(lowest-entropy) views that Top-K TTA keeps — tying Figure 1 to the Top-K strategy.

In [ ]:
import os
from IPython.display import IFrame, display
strip_dir = 'figures/strip'
strips = sorted(f for f in os.listdir(strip_dir) if f.startswith(f'{DATASET}_effb0') and f.endswith('.pdf'))
print('strips:', strips)
# PDFs render via IFrame in Kaggle; the accuracy-vs-N PNG renders directly.
from IPython.display import Image
acc_png = f'figures/tta/{DATASET}_effb0_accuracy_vs_n.png'
if os.path.exists(acc_png):
    display(Image(filename=acc_png))

## 10. Download artifacts

Zip checkpoints, results, figures, and predictions for the shared Drive. The
`predictions/` folder and checkpoints are gitignored — push them to Drive, not git.

In [ ]:
!zip -r -q /kaggle/working/tta_outputs.zip checkpoints results figures predictions
print('Wrote /kaggle/working/tta_outputs.zip — download from the Output panel.')
print('Tracker: paste results/{}_effb0_weighted_tta.csv into the EfficientNet-B0 block '
      'of Sheet 3, the benchmark into Sheet 6.'.format(DATASET))